Start Counts of each microstate across cooling rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from matplotlib.ticker import AutoMinorLocator


# =============================================================================
# LOCAL DIRECTORIES
# =============================================================================

input_root = Path("./input").resolve()

output_directory = Path(
    "./output/fig-s17"
).resolve()

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# ANALYSIS SETTINGS
# =============================================================================

k_choice = 50
tica_lag = 250

cooling_labels = [
    "6e9",
    "2e9",
    "6e8",
    "2e8",
    "6e7",
    "2e7",
    "6e6",
]


# =============================================================================
# DERIVED INPUT / OUTPUT PATHS
# =============================================================================

pairs_directory = (
    input_root
    / "cooling"
    / "pairs"
)

cooling_pair_paths = {
    cooling_label: (
        pairs_directory
        / (
            f"{cooling_label}_pairs_"
            f"t{tica_lag}_"
            f"k{k_choice}.npy"
        )
    )
    for cooling_label in cooling_labels
}

output_path = (
    output_directory
    / (
        f"start_counts_per_state_"
        f"k{k_choice}_"
        f"t{tica_lag}.pdf"
    )
)


# =============================================================================
# COLOURS AND LABELS
# =============================================================================

cooling_colors = {
    "6e9": "#fde725",
    "2e9": "#90d743",
    "6e8": "#35b779",
    "2e8": "#21918c",
    "6e7": "#31688e",
    "2e7": "#443983",
    "6e6": "#440154",
}

cooling_pretty_labels = {
    "6e9": "6.4 e9 K/s",
    "2e9": "2.7 e9 K/s",
    "6e8": "6.4 e8 K/s",
    "2e8": "2.7 e8 K/s",
    "6e7": "6.4 e7 K/s",
    "2e7": "2.7 e7 K/s",
    "6e6": "6.4 e6 K/s",
}


# =============================================================================
# FIGURE GEOMETRY
# =============================================================================

DATA_MARGIN_LEFT_IN = 1.25
DATA_MARGIN_RIGHT_IN = 0.25
DATA_MARGIN_BOTTOM_IN = 0.60
DATA_MARGIN_TOP_IN = 0.35


def make_fixed_axes_figure(
    data_width,
    data_height,
):
    fig_width = (
        DATA_MARGIN_LEFT_IN
        + data_width
        + DATA_MARGIN_RIGHT_IN
    )

    fig_height = (
        DATA_MARGIN_BOTTOM_IN
        + data_height
        + DATA_MARGIN_TOP_IN
    )

    fig = plt.figure(
        figsize=(
            fig_width,
            fig_height,
        )
    )

    ax_left = (
        DATA_MARGIN_LEFT_IN
        / fig_width
    )

    ax_bottom = (
        DATA_MARGIN_BOTTOM_IN
        / fig_height
    )

    ax_width = (
        data_width
        / fig_width
    )

    ax_height = (
        data_height
        / fig_height
    )

    ax = fig.add_axes(
        [
            ax_left,
            ax_bottom,
            ax_width,
            ax_height,
        ]
    )

    return fig, ax


# =============================================================================
# AXIS FORMATTING
# =============================================================================

def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
):
    line_width_pts = (
        stroke_in * 72
    )

    ax.minorticks_on()

    if ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.xaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_x
            )
        )

    if ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.yaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_y
            )
        )

    for spine in ax.spines.values():
        spine.set_linewidth(
            1.5
        )
        spine.set_color(
            "black"
        )

    ax.tick_params(
        which="major",
        width=line_width_pts,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width_pts * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(
        label_fontsize
    )

    ax.yaxis.label.set_size(
        label_fontsize
    )

    for tick_label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):
        tick_label.set_fontsize(
            tick_labelsize
        )

    ax.title.set_fontsize(
        title_fontsize
    )


# =============================================================================
# VALIDATE INPUT FILES
# =============================================================================

missing_files = [
    path
    for path in cooling_pair_paths.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing cooling-pair files:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_files
        )
    )


# =============================================================================
# LOAD START COUNTS
# =============================================================================

start_count_data = {}

for cooling_label in cooling_labels:
    file_path = cooling_pair_paths[
        cooling_label
    ]

    pairs = np.load(
        file_path,
        allow_pickle=True,
    )

    pairs_array = np.asarray(
        pairs
    )

    if (
        pairs_array.ndim == 2
        and pairs_array.shape[1] >= 2
    ):
        start_states = pairs_array[
            :,
            0,
        ].astype(
            int
        )

    elif pairs_array.ndim == 1:
        start_states = np.asarray(
            [
                item[0]
                for item in pairs_array
            ],
            dtype=int,
        )

    else:
        raise ValueError(
            f"Unexpected cooling-pair shape in "
            f"{file_path}:\n"
            f"{pairs_array.shape}"
        )

    invalid_states = (
        (start_states < 0)
        | (start_states >= k_choice)
    )

    if np.any(
        invalid_states
    ):
        bad_indices = np.where(
            invalid_states
        )[0][
            :10
        ]

        raise ValueError(
            f"Start states in {file_path.name} fall "
            f"outside 0..{k_choice - 1}.\n"
            f"First invalid rows: "
            f"{bad_indices.tolist()}\n"
            f"Values: "
            f"{start_states[bad_indices].tolist()}"
        )

    start_counts = np.bincount(
        start_states,
        minlength=k_choice,
    )[
        :k_choice
    ]

    start_count_data[
        cooling_label
    ] = start_counts

    print(
        f"{cooling_label:>3}  "
        f"total={start_counts.sum():>6d}  "
        f"min={start_counts.min():>4d}  "
        f"mean={start_counts.mean():>7.2f}  "
        f"max={start_counts.max():>4d}"
    )


# =============================================================================
# BUILD FIGURE
# =============================================================================

states = np.arange(
    k_choice
)

fig, ax = make_fixed_axes_figure(
    data_width=8.0,
    data_height=5.0,
)

bbox = ax.get_position()

fig_width_in, fig_height_in = (
    fig.get_size_inches()
)

axes_width_in = (
    bbox.width
    * fig_width_in
)

axes_height_in = (
    bbox.height
    * fig_height_in
)

print(
    f"Axes size: "
    f"{axes_width_in:.2f} in x "
    f"{axes_height_in:.2f} in"
)


# =============================================================================
# BARS
# =============================================================================

bar_width = 0.11

offsets = (
    np.arange(
        len(
            cooling_labels
        )
    )
    - (
        len(
            cooling_labels
        )
        - 1
    )
    / 2.0
) * bar_width

for (
    offset,
    cooling_label,
) in zip(
    offsets,
    cooling_labels,
):
    ax.bar(
        states + offset,
        start_count_data[
            cooling_label
        ],
        width=bar_width,
        color=cooling_colors[
            cooling_label
        ],
        label=cooling_pretty_labels[
            cooling_label
        ],
    )


# =============================================================================
# AXES
# =============================================================================

ax.set_xlabel(
    "State"
)

ax.set_ylabel(
    "Start count"
)

ax.set_xlim(
    -0.75,
    k_choice - 0.25,
)


# =============================================================================
# TICK LOCATORS
# =============================================================================

# Choose a sensible major state interval from k_choice.
if k_choice <= 20:
    x_major_interval = 5
elif k_choice <= 100:
    x_major_interval = 10
elif k_choice <= 250:
    x_major_interval = 25
else:
    x_major_interval = 50

ax.xaxis.set_major_locator(
    ticker.MultipleLocator(
        x_major_interval
    )
)

ax.xaxis.set_minor_locator(
    ticker.MultipleLocator(
        1
    )
)


# Dynamically choose clean y-axis major/minor spacing.
maximum_count = max(
    int(
        counts.max()
    )
    for counts in start_count_data.values()
)

if maximum_count <= 100:
    y_major_interval = 20
elif maximum_count <= 250:
    y_major_interval = 50
elif maximum_count <= 500:
    y_major_interval = 100
elif maximum_count <= 1000:
    y_major_interval = 200
else:
    y_major_interval = 500

y_minor_interval = (
    y_major_interval / 2
)

ax.yaxis.set_major_locator(
    ticker.MultipleLocator(
        y_major_interval
    )
)

ax.yaxis.set_minor_locator(
    ticker.MultipleLocator(
        y_minor_interval
    )
)


# =============================================================================
# FORMATTING
# =============================================================================

format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=12,
    tick_labelsize=11,
    title_fontsize=10,
    stroke_in=0.01,
)


# Re-apply explicit locators because format_axes sets AutoMinorLocator.
ax.xaxis.set_major_locator(
    ticker.MultipleLocator(
        x_major_interval
    )
)

ax.xaxis.set_minor_locator(
    ticker.MultipleLocator(
        1
    )
)

ax.yaxis.set_major_locator(
    ticker.MultipleLocator(
        y_major_interval
    )
)

ax.yaxis.set_minor_locator(
    ticker.MultipleLocator(
        y_minor_interval
    )
)


# =============================================================================
# LEGEND
# =============================================================================

ax.legend(
    loc="upper right",
    frameon=False,
    fontsize=12,
)


# =============================================================================
# SAVE
# =============================================================================

fig.savefig(
    output_path,
    format="pdf",
)

print(
    f"Saved figure to: "
    f"{output_path}"
)

plt.show()